# safety: limits, workspace, e-stop

Library notebook for `gambit_arm`. Every joint command passes through `validate_command`
before it reaches a backend (real servos, Gazebo, LeRobot). Limits come from
`gambit_bringup/config/safety.yaml`; the same file is meant to drive the C++ daemon and the
STM32 firmware (`firmware/stm32/Core/Src/limits.c`), which enforces them again in real time.

Milestone: *Hardware and execution* (safe operation is a precondition for every trial).

**TODO**
- [x] load safety.yaml, joint limits with margin, clamp and validate joint commands
- [x] velocity / step-size check between consecutive commands
- [x] workspace box and tilt check on the FK tool pose
- [x] latched software e-stop with heartbeat timeout and over-current hold time
- [ ] self-collision / table-collision check along the whole segment, not just endpoints
      (sample FK between waypoints, check every intermediate frame's height)
- [ ] measure real limits (safety.yaml says "measure/verify") and tighten them
- [ ] wire `EStop` to the servo bus: on trip, hold position or disable torque per `estop.on_trip`
- [ ] a physical e-stop button (cuts servo power); software e-stop is not a substitute

In [ ]:
import time
from dataclasses import dataclass, field
import numpy as np

%run ./kinematics.ipynb

SAFETY = load_yaml("safety.yaml")
list(SAFETY)

## Joint limits

URDF limits (copied into safety.yaml), shrunk by `joint_margin` so we never drive a servo
into its end stop. `clamp_joints` is for gentle use (teleop, small corrections);
`validate_command` is the gatekeeper that *rejects* bad commands instead of silently fixing them.

In [ ]:
def joint_limits(cfg=SAFETY, names=ARM_JOINTS):
    """(lower, upper) arrays for `names`, with the margin applied."""
    m = cfg["joint_margin"]
    lo = np.array([cfg["joints"][n]["lower"] + m for n in names])
    hi = np.array([cfg["joints"][n]["upper"] - m for n in names])
    return lo, hi

def clamp_joints(q, cfg=SAFETY, names=ARM_JOINTS):
    lo, hi = joint_limits(cfg, names)
    return np.clip(np.asarray(q, float), lo, hi)

## Workspace

The gripper tip must stay inside the axis-aligned box in `workspace`, and the approach must
stay within `max_tilt_deg` of vertical when the tip is low (near pieces). Higher up (transit,
park) tilt is not checked.

In [ ]:
def in_workspace(p, cfg=SAFETY):
    w = cfg["workspace"]
    return all(lo <= v <= hi for v, (lo, hi) in zip(p, (w["x"], w["y"], w["z"])))

def check_pose(q, cfg=SAFETY, low_z=None):
    """List of problems with the FK tool pose of q (empty = fine)."""
    problems = []
    T = fk(q)
    p = T[:3, 3]
    if not in_workspace(p, cfg):
        problems.append(f"workspace: tip {np.round(p, 3).tolist()} outside box")
    low_z = BOARD["board"]["thickness"] + BOARD["arm"]["hover_height"] if low_z is None else low_z
    if p[2] < low_z:
        tilt = tilt_from_vertical_deg(q)
        if tilt > cfg["workspace"]["max_tilt_deg"]:
            problems.append(f"tilt: {tilt:.1f} deg near the board (max {cfg['workspace']['max_tilt_deg']})")
    return problems

## Validating a command

Checks, in order: shape and finiteness, joint limits, step size and velocity versus the
previous command (if given), then the FK pose. Returns a list of human-readable problems.
`assert_safe` raises, for code paths that must stop.

In [ ]:
class SafetyViolation(RuntimeError):
    pass

def validate_command(q, q_prev=None, dt=None, cfg=SAFETY, check_fk=True):
    q = np.asarray(q, float)
    if q.shape != (len(ARM_JOINTS),) or not np.all(np.isfinite(q)):
        return [f"shape/finite: expected {len(ARM_JOINTS)} finite angles, got {q}"]
    problems = []
    lo, hi = joint_limits(cfg)
    for n, v, a, b in zip(ARM_JOINTS, q, lo, hi):
        if not a <= v <= b:
            problems.append(f"joint {n}: {v:.3f} outside [{a:.3f}, {b:.3f}]")
    if q_prev is not None:
        step = np.abs(q - np.asarray(q_prev, float))
        if step.max() > cfg["velocity"]["max_step_rad"]:
            problems.append(f"step: {step.max():.3f} rad > max_step_rad")
        if dt is not None and dt > 0 and (step / dt).max() > cfg["velocity"]["max_joint_rad_s"]:
            problems.append(f"velocity: {(step / dt).max():.2f} rad/s > max_joint_rad_s")
    if check_fk:
        problems += check_pose(q, cfg)
    return problems

def assert_safe(q, **kw):
    problems = validate_command(q, **kw)
    if problems:
        raise SafetyViolation("; ".join(problems))
    return np.asarray(q, float)

def gripper_command_ok(g, cfg=SAFETY):
    j = cfg["joints"]["gripper"]
    return j["lower"] <= g <= j["upper"]

## E-stop

A latched state machine: any trip stays tripped until `clear()` (if `require_manual_reset`).
`feed_heartbeat()` must be called by the host loop; `check()` trips if it goes stale. Servo
telemetry (current, temperature) trips only after staying over the limit for
`overcurrent_hold_s`, so a short spike while grasping does not stop the game.

In [ ]:
@dataclass
class EStop:
    cfg: dict = field(default_factory=lambda: SAFETY)
    clock: object = time.monotonic
    tripped: bool = False
    reasons: list = field(default_factory=list)
    _last_heartbeat: float = None
    _over_since: dict = field(default_factory=dict)

    def trip(self, reason):
        """Latch the e-stop. `reason` = "<cause>: details", cause from estop.trip_on in safety.yaml."""
        self.tripped = True
        self.reasons.append(reason)
        return self.tripped

    def clear(self):
        """Operator reset. TODO: re-read servo state and refuse to clear while a cause persists."""
        self.tripped, self.reasons = False, []
        self._over_since.clear()
        self._last_heartbeat = self.clock()

    def feed_heartbeat(self):
        self._last_heartbeat = self.clock()

    def check(self):
        """Call every control cycle. Returns True if motion is allowed."""
        now = self.clock()
        if self._last_heartbeat is not None and now - self._last_heartbeat > self.cfg["estop"]["heartbeat_timeout_s"]:
            self.trip("heartbeat_lost: no heartbeat")
        return not self.tripped

    def update_telemetry(self, currents_ma=None, temps_c=None):
        """currents_ma / temps_c: {joint: value}. Trips on sustained over-current or over-temperature."""
        now = self.clock()
        hold = self.cfg["torque"]["overcurrent_hold_s"]
        for j, i_ma in (currents_ma or {}).items():
            limit = self.cfg["torque"]["max_current_ma"]
            if abs(i_ma) > limit:
                start = self._over_since.setdefault(j, now)
                if now - start >= hold:
                    self.trip(f"overcurrent: {j} {i_ma:.0f} mA > {limit} for {hold}s")
            else:
                self._over_since.pop(j, None)
        for j, t in (temps_c or {}).items():
            if t > self.cfg["torque"]["max_temperature_c"]:
                self.trip(f"overtemperature: {j} {t} C")
        return not self.tripped

## Self-check

In [ ]:
lo, hi = joint_limits()
assert np.all(lo < hi) and np.isclose(hi[0], 1.91986 - SAFETY["joint_margin"])
assert np.allclose(clamp_joints([5, -5, 0, 0, 0])[:2], [hi[0], lo[1]])
q_e4, info = ik_top_down(hover_point("e4"))
assert info["converged"] and validate_command(q_e4) == [], validate_command(q_e4)
assert any("joint shoulder_pan" in p for p in validate_command([3.0, 0, 0, 0, 0], check_fk=False))
assert any(p.startswith("step") for p in validate_command(q_e4, q_prev=q_e4 + 0.5))
assert any(p.startswith("velocity") for p in validate_command(q_e4, q_prev=q_e4 + 0.1, dt=0.01))
assert validate_command([np.nan] * 5)[0].startswith("shape")
assert in_workspace(hover_point("a1")) and in_workspace(hover_point("h8"))
assert in_workspace(graveyard_slot(3) + [0, 0, 0.01]) and not in_workspace([0.0, 0.0, 0.5])
try:
    assert_safe([3.0, 0, 0, 0, 0], check_fk=False); raise AssertionError("should raise")
except SafetyViolation:
    pass
assert gripper_command_ok(BOARD["arm"]["gripper_open"]) and gripper_command_ok(BOARD["arm"]["gripper_closed"])
# e-stop with a fake clock
_t = [0.0]
es = EStop(clock=lambda: _t[0])
es.feed_heartbeat(); _t[0] = 0.1
assert es.check()
es.update_telemetry({"elbow_flex": 5000}); _t[0] = 0.2
assert es.update_telemetry({"elbow_flex": 5000})            # 0.1 s over: still OK
_t[0] = 0.5; es.feed_heartbeat()
assert not es.update_telemetry({"elbow_flex": 5000})        # 0.4 s over: tripped, latched
es.update_telemetry({"elbow_flex": 0})
assert not es.check() and "overcurrent" in es.reasons[0]
es.clear(); assert es.check()
_t[0] = 2.0
assert not es.check() and es.reasons[0].startswith("heartbeat_lost")
print("safety OK")